# CSV to SQLite Conversion
This notebook reads `g41_Publishers_Magazines_v2.csv`, normalizes data, and writes it to `data/publishers_magazines.db`.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

CSV_PATH = '../g41_Publishers_Magazines_v2.csv'
OUT_DIR = Path('data')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUT_DIR / 'publishers_magazines.db'

# Use latin-1 because the CSV includes accented characters
df = pd.read_csv(CSV_PATH, sep=';', dtype=str, encoding='latin-1').fillna('')
print(f'Rows loaded: {len(df)}')
df.head()

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Create tables
cur.execute('''
CREATE TABLE IF NOT EXISTS publishers (
  publisher_id INTEGER PRIMARY KEY,
  publisher_name TEXT,
  created_date TEXT
)''')

cur.execute('''
CREATE TABLE IF NOT EXISTS magazines (
  magazine_id INTEGER PRIMARY KEY,
  magazine_title TEXT,
  magazine_category TEXT
)''')

cur.execute('''
CREATE TABLE IF NOT EXISTS warehouses (
  warehouses_id INTEGER PRIMARY KEY,
  warehouses_info TEXT
)''')

cur.execute('''
CREATE TABLE IF NOT EXISTS transactions (
  transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
  transaction_date TEXT,
  amount REAL,
  publisher_id INTEGER,
  magazine_id INTEGER,
  warehouses_id INTEGER,
  FOREIGN KEY(publisher_id) REFERENCES publishers(publisher_id),
  FOREIGN KEY(magazine_id) REFERENCES magazines(magazine_id),
  FOREIGN KEY(warehouses_id) REFERENCES warehouses(warehouses_id)
)''')

conn.commit()
print('Tables created.')

In [ ]:
# Insert unique publishers
publishers = df[['publisher_id', 'publisher_name', 'created_date']].drop_duplicates().copy()
publishers['publisher_id'] = publishers['publisher_id'].apply(lambda x: int(x) if x != '' else None)
publishers.to_sql('publishers', conn, if_exists='replace', index=False)

# Insert unique magazines
magazines = df[['magazine_id', 'magazine_title', 'magazine_category']].drop_duplicates().copy()
magazines['magazine_id'] = magazines['magazine_id'].apply(lambda x: int(x) if x != '' else None)
magazines.to_sql('magazines', conn, if_exists='replace', index=False)

# Insert unique warehouses (ignore empty ids)
def parse_warehouse_id(value):
    try:
        return int(value) if value not in ('', '0') else None
    except Exception:
        return None

warehouses = df[['warehouses_id', 'warehouses_info']].drop_duplicates().copy()
warehouses['warehouses_id'] = warehouses['warehouses_id'].apply(parse_warehouse_id)
warehouses = warehouses[warehouses['warehouses_id'].notnull()].copy()
warehouses.to_sql('warehouses', conn, if_exists='replace', index=False)

# Insert transactions
transactions = df[['transaction_date', 'amount', 'publisher_id', 'magazine_id', 'warehouses_id']].copy()
transactions['amount'] = transactions['amount'].apply(lambda x: float(x) if x != '' else 0.0)
transactions['publisher_id'] = transactions['publisher_id'].apply(lambda x: int(x) if x != '' else None)
transactions['magazine_id'] = transactions['magazine_id'].apply(lambda x: int(x) if x != '' else None)
transactions['warehouses_id'] = transactions['warehouses_id'].apply(parse_warehouse_id)
transactions.to_sql('transactions', conn, if_exists='replace', index=False)

conn.commit()
conn.close()
print(f'Database created at: {DB_PATH}')